# ETF估值与指数调仓来源资格
冻结12次请求，仅做来源资格，不读取收益、不拟合、不回测。主假说是估值与盈利周期能否增加月度配置的信息，调仓支线独立；本卡只决定输入是否可用。

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
while not (root / 'SPEC.md').exists(): root = root.parent
sys.path.insert(0, str(root))
import qlib
from qlib.data import D
qlib.init(provider_uri=str(root/'data/qlib/soxx_pit_leveraged_training_v1'), region='us', kernels=1)
calendar = D.calendar(start_time='2019-01-01', end_time='2026-10-16')
from src.data.etf_valuation_source import run
summary = run(root)
summary

[73974:MainThread](2026-10-16 11:28:00,237) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


2026/10/16 11:28:00 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[73974:MainThread](2026-10-16 11:28:00,612) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[73974:MainThread](2026-10-16 11:28:00,613) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxx_pit_leveraged_training_v1')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[73974:MainThread](2026-10-16 11:28:01,364) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[73974:MainThread](2026-10-16 11:28:01,365) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[73974:MainThread](2026-10-16 11:28:01,365) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxx_pit_leveraged_training_v1')}


{'experiment': 'etf_valuation_source_20261016',
 'decision': 'needs_qualified_historical_valuation_inputs',
 'objective_achieved': False,
 'new_fits': 0,
 'new_backtests': 0,
 'qualified_monthly_rows': 0,
 'qualified_input_path': None,
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/soxx_pit_leveraged_training_v1',
 'market': 'SOXX',
 'region': 'us',
 'as_of': '2026-10-16',
 'qlib_calendar_rows': 1885,
 'calendar_first': '2019-01-02 00:00:00',
 'calendar_last': '2026-07-02 00:00:00',
 'holdings_rows': 840,
 'historic_symbols': 47,
 'holdings_filings': 28,
 'max_disclosure_lag_days': 286,
 'fundamentals_inventory': [{'ticker': 'NVDA',
   'fact_rows': 871,
   'accessions': 70,
   'filed_start': '2009-08-20',
   'filed_end': '2026-08-26',
   'status': 'candidate_accession_facts_need_original_filing_period_share_basis_validation',
   'original_period_candidates': 320,
   'first_filing_conflicts': 0},
  {'ticker': 'INTC',
   'fact_rows': 712,
   'accessions': 70,
   'filed_start': 

In [2]:
from qlib.workflow import R
R.set_uri(summary['recorder_uri'])
rec = R.get_recorder(recorder_id=summary['recorder_id'], experiment_id=summary['experiment_id'])
rec.load_object('analysis/source_qualification')

,source,status,qualified_months,detail
0,issuer historical workbook probes,needs_historical_ratio_publication_vintages,0,"HTTP/body, date and ratio availability in down..."
1,SEC NPORT local canonical history,candidate_delayed_disclosed_basket,0,840 holdings; 47 symbols; 28 filings; no exact...
2,existing NVDA earnings corpus,partial_basket_only,0,NVDA weight 6.7918 to 11.0275 percent; cannot ...
3,SEC companyfacts representative probe,candidate_reconstruction_intake,0,See fundamentals_inventory.csv; original fisca...
4,SOXX versus SOX methodology,context_only_needs_dated_weight_files,0,2021 index change and 2026 method separately r...


# ETF来源资格结论

实际完成12次限定请求、Qlib日历读取和原生Recorder登记。3个历史日期的旧下载接口均返回HTML；新接口均返回同一2026-10-14持仓文件，没有历史P/E或P/B字段，因此不能用请求参数伪造历史估值。当前页面估值只作当前观察。

已有NPORT提供840条持仓、47个历史代码和28次披露，权益覆盖99.70%–99.88%。末次2026-07-13披露实际指向旧2025-09-30报告，必须避免最新披露覆盖较新报告的倒退；报告日至披露最长286天。报告期与可用时点不能互换。季度披露本身不能满足新鲜月度估值和36个独立月度观察要求。

SEC companyfacts免费入口可用：NVDA871条、INTC712条相关事实；保留首次申报候选320/325条，后续比较期修订不覆盖原值。原始NVDA2026年第二季10-Q文本亦已下载，收入、EPS标签与facts一致；这不能认证其他发行人或全部历史的原始性。估值重建需要TTM、股数/ADR/拆股及价格基准，并纳入负盈利；不能简单平均P/E，不能把未覆盖的亏损权重写为零。

SOXX在2021年的PHLX→ICE公告与当前NYSE标签分别登记，尚未确认ICE/NYSE之间完整版本沿革。SOX当前2026方法不能回填历史；公告日期和生效日期分开，缺历史权重文件时不生成调仓信号。SOXX总收益只能与明确的指数总收益版本比较。

本卡没有训练或回测，合格月度估值行数0，不晋升，不代表稳定超额失败或全局阻塞。下一步已启动另一冻结卡的47代码历史发行人事实采集；尚未获得历史估值分位或策略超额结论。不同年份/指数版本只有来源覆盖差异，未做行情窗口优劣选择。

证据：summary.json、download_audit.json、source_qualification.csv、holdings_coverage.csv、fundamentals_inventory.csv、methodology_sources.json；完整原始响应和事实表在Recorder及NAS归档。
